# NB06 - Evaluation: MRI vs baselines, calibration, conformal prediction, age control
All numbers come from **pooled out-of-fold predictions** (each participant is scored by models that never saw them).
Means over CV repeats with participant-level bootstrap 95 % CIs.

In [ ]:
# ── 1. Environment Detection (Google Colab vs Local Machine) ──────────────
import sys, os, subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── 2. Configure Paths ───────────────────────────────────────────────────
if IN_COLAB:
    # Google Colab paths (on your Google Drive)
    PROJECT  = '/content/drive/MyDrive/oasis_project'      # this unzipped project folder on Drive
    DATA_DIR = '/content/drive/MyDrive/OASIS1'             # folder with the 12 OASIS-1 disc archives (+ clinical csv)
    OUT_DIR  = '/content/drive/MyDrive/OASIS1_outputs'     # results are saved here (survives Colab disconnects)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{PROJECT}/requirements.txt'], check=False)
else:
    # Local machine paths (runs directly on your laptop)
    PROJECT  = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
    DATA_DIR = os.path.join(PROJECT, 'Dataset')
    OUT_DIR  = os.path.join(PROJECT, 'OASIS1_outputs')

# ── 3. install + load ─────────────────────────────────────────────────────
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
from oasis_ad.config import load_config, Paths
from oasis_ad import pipeline as PL, DISCLAIMER
cfg = load_config(os.path.join(PROJECT, 'configs', 'default.yaml'),
                  {'paths.data_dir': DATA_DIR, 'paths.out_dir': OUT_DIR})
paths = Paths(cfg)
import torch, pandas as pd
print(DISCLAIMER)
print('Environment:', 'Google Colab' if IN_COLAB else 'Local Laptop')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (on Colab switch to Runtime > Change runtime type > T4 GPU)')

In [ ]:
MODEL = 'cnn3d'
res = PL.step6_evaluate(cfg, paths, MODEL)
res['model_comparison']

### Does the image add information beyond age? (RQ2)

In [ ]:
print('Age-matched subset:'); display(res['age_matched'])
print('Score vs age (Spearman):', res['age_dependence'])
print('CV AUROC with age vs age + MRI score:', res['incremental_over_age'])
print('Paired bootstrap AUROC difference (MRI - baseline):'); import json; print(json.dumps(res['paired_vs_baselines'], indent=2))

### Subgroup performance (age bands, sex)

In [ ]:
cols = [c for c in ['model','subgroup','n','n_positive','balanced_acc','balanced_acc_CI','auroc','auroc_CI','sensitivity','specificity','note'] if c in res['subgroups'].columns]
display(res['subgroups'][cols].fillna(''))

### Calibration (RQ4): raw softmax vs temperature scaling vs MC dropout vs deep ensemble of the CV repeats

In [ ]:
display(res['calibration'])
from IPython.display import Image
display(Image(f'{paths.evaluation}/{MODEL}/reliability.png'))
print(res['uncertainty_vs_error'])

### Conformal prediction (RQ5) - coverage must be checked per subgroup

In [ ]:
c = res['conformal']; print('target coverage', c['target_coverage'], ' 95% CI of observed coverage', c['overall_coverage_CI95'])
pd.DataFrame(c['pooled_mean_over_repeats']).T